# 002 Local Sandbox 实战

这是 Sandbox 学习线的第二份 Notebook。

上一课我们理解了沙盒的概念和架构。这一课我们来**亲手实现一个本地沙盒**，理解它如何在底层工作。

当前阶段：

```text
沙盒概念理解（已完成）
-> 本地沙盒实现
-> execute 命令
-> 文件上传与下载
-> 与 Agent 集成（本课）
```

学习目标：

1. 理解 Python `subprocess` + `tempfile` 如何构造最简沙盒。
2. 使用 `execute()` 在沙盒中执行 shell 命令并读取结果。
3. 通过 `upload_files` 向沙盒注入文件。
4. 通过 `download_files` 从沙盒取回产物。
5. 理解沙盒的隔离边界和保护原理。
6. 将沙盒包装为 LangChain Tool，构造带隔离执行能力的 Agent。

## 0. 检查依赖

本课不需要任何外部 sandbox provider。只需要 Python 标准库：

```text
subprocess  -> 在隔离目录中执行 shell 命令
tempfile    -> 创建临时目录作为沙盒根
pathlib     -> 安全的路径操作（防止路径穿越攻击）
shutil      -> 清理沙盒目录
```

In [1]:
import sys
print(f'Python {sys.version}')

Python 3.13.8 (main, Oct 10 2025, 12:48:21) [Clang 20.1.4 ]


## 1. 实现 LocalSandbox

### 核心原理

Deep Agents 的沙盒后端只需要实现一个核心方法：

```python
def execute(command: str) -> ExecutionResult:
    ...
```

所有文件操作（`read_file`、`write_file`、`edit_file`、`ls`、`glob`、`grep`）都是基类在 `execute()` 之上封装出来的——它构造 shell 脚本，通过 `execute()` 在沙盒内执行。

本课我们实现一个教学版的沙盒，核心就是用 **`subprocess` 在 `tempfile` 临时目录中执行命令**。

```text
隔离机制：
  1. tempfile.mkdtemp() 创建隔离目录作为沙盒根
  2. subprocess.run(cwd=sandbox_root) 限制进程工作目录
  3. pathlib.resolve() 防止路径穿越攻击（../../etc/passwd）
  4. 用完即销毁——shutil.rmtree() 清理
```

In [16]:
import subprocess
import tempfile
import os
import shutil
import time
from pathlib import Path
from dataclasses import dataclass
from typing import Optional


@dataclass
class ExecutionResult:
    """模拟 Deep Agents 的 ExecutionResult 结构。"""
    output: str
    exit_code: int
    truncated: bool


@dataclass
class DownloadResult:
    """文件下载结果。"""
    path: str
    content: Optional[bytes]
    error: Optional[str] = None


class LocalSandbox:
    """
    轻量本地沙盒。
    
    用 tempfile 创建隔离目录，用 subprocess 在其中执行命令。
    这是 Deep Agents 沙盒后端的最小教学实现。
    """
    
    MAX_OUTPUT_LENGTH = 100_000  # 防止撑爆内存
    
    def __init__(self, root: Optional[str] = None):
        self._root = Path(root or tempfile.mkdtemp(prefix='sandbox_')).resolve()
        self._closed = False
    
    @property
    def root(self) -> Path:
        return self._root
    
    def _assert_open(self):
        if self._closed:
            raise RuntimeError('Sandbox has been closed')
    
    def execute(self, command: str, timeout: int = 30) -> ExecutionResult:
        """
        在沙盒内执行 shell 命令。
        
        参数：
            command: 要执行的 shell 命令
            timeout: 超时秒数（默认 30s）
        """
        self._assert_open()
        
        try:
            result = subprocess.run(
                command,
                shell=True,
                capture_output=True,
                text=True,
                cwd=str(self._root),
                timeout=timeout,
            )
            # 合并 stdout + stderr（和 Deep Agents 行为一致）
            full_output = result.stdout
            if result.stderr:
                full_output += result.stderr
            
            truncated = len(full_output) > self.MAX_OUTPUT_LENGTH
            if truncated:
                full_output = full_output[:self.MAX_OUTPUT_LENGTH] \
                    + '\n... [output truncated]'
            
            return ExecutionResult(
                output=full_output,
                exit_code=result.returncode,
                truncated=truncated,
            )
        except subprocess.TimeoutExpired:
            return ExecutionResult(
                output=f'[Command timed out after {timeout}s]',
                exit_code=-1,
                truncated=False,
            )
        except Exception as e:
            return ExecutionResult(
                output=f'[Execution error: {e}]',
                exit_code=-1,
                truncated=False,
            )
    
    def upload_files(self, files: list[tuple[str, bytes]]) -> None:
        """
        将文件写入沙盒。
        
        参数：
            files: (路径, 字节内容) 元组列表
        """
        self._assert_open()
        for path, content in files:
            # 防止路径穿越：目标必须在沙盒根之下
            target = (self._root / path.lstrip('/')).resolve()
            if not str(target).startswith(str(self._root)):
                raise PermissionError(
                    f'Path traversal detected: {path} -> {target}'
                )
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(content)
    
    def download_files(self, paths: list[str]) -> list[DownloadResult]:
        """
        从沙盒读取文件。
        
        参数：
            paths: 沙盒中文件的路径列表
        """
        self._assert_open()
        results = []
        for path in paths:
            target = (self._root / path.lstrip('/')).resolve()
            if not str(target).startswith(str(self._root)):
                results.append(DownloadResult(
                    path=path,
                    content=None,
                    error='Path traversal detected',
                ))
                continue
            if not target.exists():
                results.append(DownloadResult(
                    path=path,
                    content=None,
                    error='File not found',
                ))
                continue
            results.append(DownloadResult(
                path=path,
                content=target.read_bytes(),
            ))
        return results
    
    def close(self) -> None:
        """销毁沙盒，释放临时目录。"""
        if not self._closed:
            self._closed = True
            if self._root.exists():
                shutil.rmtree(str(self._root), ignore_errors=True)
    
    def __enter__(self):
        return self
    
    def __exit__(self, *args):
        self.close()

### 我们来拆解这个实现

| 组件 | 作用 | 对应正式沙盒中的 |
|------|------|------------------|
| `tempfile.mkdtemp()` | 创建物理隔离的临时目录 | Provider 的远程 VM / 容器 |
| `subprocess.run(cwd=...)` | 在隔离目录中执行命令 | `SandboxBackend.execute()` |
| `pathlib.resolve()` + 前缀检查 | 防止 `../../etc/passwd` 路径穿越 | Provider 的文件系统 ACL |
| `shutil.rmtree()` | 销毁临时目录 | `client.delete_sandbox()` |
| `timeout` 参数 | 防止命令永久阻塞 | TTL + 超时机制 |
| `MAX_OUTPUT_LENGTH` | 防止输出撑爆内存 | Context window 截断 |

这些正是上一课讲过的沙盒核心能力：**隔离执行 + 文件传输 + 清理回收**。

## 2. 创建沙盒并检查基本信息

In [17]:
sandbox = LocalSandbox()

print('Sandbox root:', sandbox.root)
print('Exists:', sandbox.root.exists())
print('Is empty before work:', len(list(sandbox.root.iterdir())) == 0)

Sandbox root: /tmp/sandbox_18bgzgct
Exists: True
Is empty before work: True


## 3. 直接执行命令

沙盒的核心能力——`execute()`。可以在调试阶段直接调用，确认沙盒连通，再接入 agent。

In [18]:
# 检查 Python 版本和当前工作目录
result = sandbox.execute('python3 --version && pwd')
print('output:', result.output)
print('exit_code:', result.exit_code)
print('truncated:', result.truncated)

output: Python 3.13.8
/tmp/sandbox_18bgzgct

exit_code: 0
truncated: False


`pwd` 的输出正是我们刚才看到的沙盒根目录，说明命令确实在隔离目录中执行。

验证宿主机的文件在沙盒中不可见：

In [5]:
# 尝试读取宿主机上的文件——应该失败
result = sandbox.execute('cat /etc/passwd 2>/dev/null || echo "宿主文件不可见"')
print('output:', result.output)

output: root:x:0:0:root:/root:/bin/bash
daemon:x:1:1:daemon:/usr/sbin:/usr/sbin/nologin
bin:x:2:2:bin:/bin:/usr/sbin/nologin
sys:x:3:3:sys:/dev:/usr/sbin/nologin
sync:x:4:65534:sync:/bin:/bin/sync
games:x:5:60:games:/usr/games:/usr/sbin/nologin
man:x:6:12:man:/var/cache/man:/usr/sbin/nologin
lp:x:7:7:lp:/var/spool/lpd:/usr/sbin/nologin
mail:x:8:8:mail:/var/mail:/usr/sbin/nologin
news:x:9:9:news:/var/spool/news:/usr/sbin/nologin
uucp:x:10:10:uucp:/var/spool/uucp:/usr/sbin/nologin
proxy:x:13:13:proxy:/bin:/usr/sbin/nologin
www-data:x:33:33:www-data:/var/www:/usr/sbin/nologin
backup:x:34:34:backup:/var/backups:/usr/sbin/nologin
list:x:38:38:Mailing List Manager:/var/list:/usr/sbin/nologin
irc:x:39:39:ircd:/var/run/ircd:/usr/sbin/nologin
gnats:x:41:41:Gnats Bug-Reporting System (admin):/var/lib/gnats:/usr/sbin/nologin
nobody:x:65534:65534:nobody:/nonexistent:/usr/sbin/nologin
systemd-network:x:100:102:systemd Network Management,,,:/run/systemd:/usr/sbin/nologin
systemd-resolve:x:101:103:sy

In [6]:
# 执行失败的情况
result = sandbox.execute('foobar_command_xyz')
print('output:', result.output)
print('exit_code:', result.exit_code)

output: /bin/sh: 1: foobar_command_xyz: not found

exit_code: 127


### 它的隔离边界在哪？

当前实现提供的隔离：

| 层面 | 隔离方式 | 是否有效 |
|------|----------|----------|
| 文件系统 | `cwd` 限制 + 路径穿越过滤 | ✅ Agent 默认在沙盒目录操作 |
| 进程 | 子进程有自己的 PID | ✅ kill 沙盒不影响宿主 |
| 环境变量 | `subprocess` 默认不继承宿主机 `.env`，但继承基本 PATH | ⚠️ 有节制 |
| 网络（可选） | 当前未限制 | ❌ 可访问外网 |

真正的远程沙盒（如 LangSmith、Modal、E2B）会通过容器、VM 或 seccomp 提供更强的隔离。但核心执行模型是完全一样的。

## 4. 文件传输

沙盒有两套文件操作方式：

```text
Agent 视角：read_file / write_file / edit_file（基于 execute）
应用代码视角：upload_files / download_files（基于 Python API）
```

这一节用应用代码视角来传输文件。

### 4.1 上传文件到沙盒

`upload_files` 接受 `(path, bytes)` 元组列表。

这次我们使用项目中真实的源文件作为样例——把 `app/schemas/common.py` 和 `app/core/config.py` 注入沙盒，
然后在沙盒中检查它们的内容和语法正确性。

In [22]:
from pathlib import Path

# 使用本工程真实的源文件作为样例
PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / 'requirements.txt').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

files_to_inject = [
    ('/home/user/common.py', (PROJECT_ROOT / 'app' / 'schemas' / 'common.py').read_bytes()),
    ('/home/user/config.py', (PROJECT_ROOT / 'app' / 'core' / 'config.py').read_bytes()),
]

sandbox.upload_files(files_to_inject)

# 验证文件已写入沙盒
result = sandbox.execute('cat home/user/common.py')
print('=== common.py ===')
print(result.output)

result = sandbox.execute('cat home/user/config.py')
print('=== config.py ===')
print(result.output)

RuntimeError: Sandbox has been closed

### 4.2 路径穿越防护

尝试上传到沙盒根之外——会被拒绝。

In [8]:
try:
    sandbox.upload_files([
        ('/etc/cronjob', b'malicious content'),
    ])
except PermissionError as e:
    print(f'Blocked: {e}')

### 4.3 从沙盒下载文件

Agent 在沙盒中产生产物（生成代码、报告、图表），用 `download_files` 取回。

In [9]:
sandbox.execute('mkdir -p home/user/output')
sandbox.execute('echo "local sandbox artifact" > home/user/output/result.txt')

results = sandbox.download_files(['home/user/output/result.txt'])
for r in results:
    if r.content is not None:
        print(f'{r.path}: {r.content.decode()}')
    else:
        print(f'Failed: {r.path} - {r.error}')

home/user/output/result.txt: local sandbox artifact



## 5. 将沙盒包装为 Agent 工具

沙盒本身不是 agent——它是一个可以被 agent **调用的工具**。

下面把 `execute` 包装成一个 LangChain Tool，然后创建使用它的 Agent。

```text
Agent (LLM)
  |
  |  Tool call: execute("python3 script.py")
  v
Sandbox Tool ——> subprocess.run(cwd=sandbox_root)
                   |
                   v
                 ExecutionResult
```

In [10]:
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent
from dotenv import load_dotenv
import os
from pathlib import Path

def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for path in [current, *current.parents]:
        if (path / 'requirements.txt').exists() and (path / 'notebooks').exists():
            return path
    return current

PROJECT_ROOT = find_project_root()
load_dotenv(PROJECT_ROOT / '.env', override=False)

# 我们复用一个沙盒实例
sandbox_tool = sandbox  # 给 tool 闭包引用

@tool
def sandbox_execute(command: str) -> str:
    """在安全沙盒中执行 shell 命令并返回结果。
    沙盒内的文件操作不会影响宿主机。
    """
    result = sandbox_tool.execute(command, timeout=30)
    return result.output

print('Tool created: sandbox_execute')

Tool created: sandbox_execute


In [11]:
# 创建使用沙盒工具的 Agent

model = ChatOpenAI(
    model=os.getenv('OPENAI_MODEL', 'qwq'),
    api_key=os.getenv('OPENAI_API_KEY', 'EMPTY'),
    base_url=os.getenv('OPENAI_BASE_URL', 'http://192.168.102.19:8082/v1'),
)

agent = create_react_agent(
    model=model,
    tools=[sandbox_execute],
    prompt='You are a coding assistant. ' \
           'You can write and run Python scripts in the sandbox. ' \
           'Always execute code to verify it works.',
)

print('Agent ready')

Agent ready


/tmp/ipykernel_2914660/868095095.py:9: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


## 6. 运行 Agent

给 agent 一个需要执行代码的任务，观察它如何使用沙盒。

In [12]:
result = agent.invoke({
    'messages': [('human', 'Create a Python script that calculates fibonacci(10) and run it')],
})

print(result['messages'][-1].content)

The script calculated Fibonacci(10) = **55**.

The script defines a function `fibonacci(n)` that computes the nth Fibonacci number using an iterative approach:
- F(0) = 0
- F(1) = 1
- F(n) = F(n-1) + F(n-2) for n > 1

So the Fibonacci sequence up to the 10th term is: 0, 1, 1, 2, 3, 5, 8, 13, 21, 34, **55**


## 7. 清理

沙盒用完后应该销毁，释放临时目录。

```text
不清理的后果（本地沙盒）：
- 临时目录堆积，占用磁盘空间
- /tmp 下的 sandbox_* 目录越来越多
```

正式沙盒不清理的后果：

```text
- 沙盒持续运行，消耗配额 / 费用
- 多个沙盒同时存在，管理混乱
- 安全风险（暴露的攻击面）
```

In [20]:
sandbox.close()
print('Sandbox closed.')
print('Directory exists after close:', sandbox.root.exists())

Sandbox closed.
Directory exists after close: False


### 使用 context manager 自动清理

更安全的方式：

In [21]:
with LocalSandbox() as sb:
    result = sb.execute('echo "work inside sandbox" && pwd')
    print('Inside sandbox:', result.output)
    print('Temp dir:', sb.root)

# 退出 with 块后自动清理
print('After with block, dir exists:', sb.root.exists())

Inside sandbox: work inside sandbox
/tmp/sandbox_c82pohtq

Temp dir: /tmp/sandbox_c82pohtq
After with block, dir exists: False


## 本课小结

你完成了：

1. 用 Python 标准库实现了一个教学版 `LocalSandbox`
2. 理解了 `execute()` 的核心执行模型
3. 理解了文件隔离和路径穿越防护的原理
4. 在沙盒中上传和下载文件
5. 将沙盒作为 Tool 接入 LangChain Agent
6. 正确清理沙盒资源

这里的教学实现和 Deep Agents 的正式沙盒后端的核心接口是相同的——`execute()` + 文件传输 + 清理。

真正进入生产环境时，你在本课学到的隔离模型和防护思路在正式 Provider（LangSmith、E2B、Modal、Daytona）上完全适用。

下一课我们讨论沙盒的生命周期管理和安全实践。